# 📕 Bài 4 — Cây quyết định & Ensemble

**Tuần 11**

> Với dữ liệu **dạng bảng**, các model trong bài này vẫn đang đánh bại deep learning. Đừng dùng mạng neural cho file CSV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]

DATA = Path("../../../data")
if not (DATA / "churn.csv").exists():
    DATA = Path("data")

df = pd.read_csv(DATA / "churn.csv").drop(columns=["da_goi_tong_dai_huy"])  # bỏ cột rò rỉ
y = df["churn"]
X = df.drop(columns=["churn", "khach_id"])

cot_so = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cot_chu = [c for c in X.columns if c not in cot_so]

pre = ColumnTransformer([
    ("so", Pipeline([("d", SimpleImputer(strategy="median")),
                     ("s", StandardScaler())]), cot_so),
    ("chu", OneHotEncoder(handle_unknown="ignore"), cot_chu),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=42)

print(f"train {len(X_train)}  |  test {len(X_test)}")

## 1. Cây quyết định — model duy nhất người thường đọc hiểu

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

cay_nong = Pipeline([("pre", pre),
                     ("m", DecisionTreeClassifier(max_depth=3, random_state=42))])
cay_nong.fit(X_train, y_train)

ten_dac_trung = cay_nong.named_steps["pre"].get_feature_names_out()

fig, ax = plt.subplots(figsize=(15, 6))
plot_tree(cay_nong.named_steps["m"], feature_names=ten_dac_trung,
          class_names=["Ở lại", "Rời bỏ"], filled=True, fontsize=8,
          impurity=False, ax=ax)
ax.grid(visible=False)
plt.tight_layout(); plt.show()

**Cách đọc:** mỗi ô hỏi một câu hỏi `đặc_trưng <= giá_trị?` — đúng thì đi trái, sai thì đi phải. `samples` là số khách rơi vào nhánh đó, `value` là phân bố hai lớp.

> 🎯 **Đây là thứ bạn in ra đưa cho trưởng phòng kinh doanh.** Không model nào khác làm được điều này.

## 2. ⚠️ Cây rất dễ overfit

In [ ]:
from sklearn.metrics import roc_auc_score

do_sau = [1, 2, 3, 5, 8, 12, 20, None]
kq = []
for d in do_sau:
    m = Pipeline([("pre", pre),
                  ("m", DecisionTreeClassifier(max_depth=d, random_state=42))]).fit(X_train, y_train)
    kq.append({
        "do_sau": "không giới hạn" if d is None else d,
        "train": roc_auc_score(y_train, m.predict_proba(X_train)[:, 1]),
        "test": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]),
    })
kq = pd.DataFrame(kq)

x = range(len(kq))
plt.plot(x, kq["train"], marker="o", lw=2, color=MAU[0], label="train")
plt.plot(x, kq["test"], marker="o", lw=2, color=MAU[3], label="test")
plt.xticks(x, kq["do_sau"])
plt.xlabel("độ sâu tối đa của cây"); plt.ylabel("AUC")
plt.title("Cây càng sâu → train càng đẹp, test càng tệ")
plt.legend(frameon=False)
plt.show()

print(kq.round(4).to_string(index=False))

> 🔑 **Đây chính là biểu đồ overfitting** từ bài 1, giờ trên dữ liệu thật.
>
> Cây không giới hạn độ sâu đạt **AUC train ≈ 1.0** — nó học thuộc từng dòng. Nhưng AUC test **tệ hơn** cả cây sâu 3.
>
> Cây có khả năng ghi nhớ hoàn hảo. Đó vừa là sức mạnh vừa là điểm yếu chết người của nó.

**Các tham số hãm overfit:** `max_depth`, `min_samples_leaf`, `min_samples_split`, `max_leaf_nodes`, `ccp_alpha`.

## 3. Random Forest — nhiều cây yếu thành một model mạnh

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([("pre", pre),
               ("m", RandomForestClassifier(n_estimators=200, random_state=42,
                                            n_jobs=-1))]).fit(X_train, y_train)

cay_don = Pipeline([("pre", pre),
                    ("m", DecisionTreeClassifier(random_state=42))]).fit(X_train, y_train)

for ten, m in [("Cây đơn (không giới hạn)", cay_don), ("Random Forest (200 cây)", rf)]:
    tr = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    te = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    print(f"  {ten:<28} train {tr:.4f}   test {te:.4f}   chênh {tr-te:.4f}")

```
🌲 🌲 🌲 🌲 🌲  →  bỏ phiếu / trung bình  →  dự đoán
```

**Random Forest hoạt động thế nào:**
1. Mỗi cây train trên một **mẫu bootstrap** (lấy ngẫu nhiên có hoàn lại) của dữ liệu
2. Ở mỗi lần chia, cây chỉ được xem **một tập con ngẫu nhiên** các đặc trưng
3. Kết quả cuối = trung bình xác suất của tất cả các cây

**Vì sao nó hoạt động:** từng cây vẫn overfit, nhưng chúng overfit theo những cách **khác nhau**. Khi lấy trung bình, lỗi ngẫu nhiên triệt tiêu lẫn nhau, còn lại tín hiệu chung.

> Đây là ý tưởng đẹp nhất của classical ML: **kết hợp nhiều model yếu và đa dạng thì tốt hơn một model mạnh.**

Chú ý cột "chênh": Random Forest **vẫn** có train cao hơn test, nhưng khoảng cách nhỏ hơn nhiều và điểm test cao hơn hẳn.

## 4. Gradient Boosting — cây sau sửa lỗi cây trước

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

cac_model = {
    "Baseline":          DummyClassifier(strategy="most_frequent"),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree(d=5)":  DecisionTreeClassifier(max_depth=5, random_state=42),
    "RandomForest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting":   HistGradientBoostingClassifier(random_state=42),
}

dong = []
for ten, m in cac_model.items():
    d = cross_val_score(Pipeline([("pre", pre), ("m", m)]), X_train, y_train,
                        cv=cv, scoring="roc_auc")
    dong.append({"model": ten, "auc_mean": round(d.mean(), 4), "auc_std": round(d.std(), 4)})

bang = pd.DataFrame(dong).sort_values("auc_mean", ascending=False).reset_index(drop=True)
print(bang.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(bang["model"][::-1], bang["auc_mean"][::-1],
        xerr=bang["auc_std"][::-1], color=MAU_CHINH, height=.6,
        error_kw={"ecolor": "#444", "capsize": 3})
ax.axvline(0.5, color="gray", ls=":", label="đoán mò")
ax.set_xlim(0.4, 0.9); ax.set_xlabel("ROC-AUC (cross-validation)")
ax.set_title("So sánh model — thanh lỗi là độ lệch chuẩn giữa các fold")
ax.grid(axis="y", visible=False); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

> 🔍 **Kết quả có thể làm bạn ngạc nhiên: LogisticRegression thắng.**
>
> Vì dữ liệu này được sinh ra từ một quan hệ **logistic** thật — đúng dạng mà model tuyến tính nắm bắt hoàn hảo. Các model phức tạp hơn phải *học lại* quan hệ đó từ dữ liệu có nhiễu, nên thua.
>
> **Bài học:** *"phức tạp hơn" không đồng nghĩa "tốt hơn"*. Luôn thử model đơn giản trước.

**Nguyên tắc đọc bảng so sánh:**

| Tình huống | Kết luận |
|---|---|
| Chênh lệch **nhỏ hơn** độ lệch chuẩn | Hai model **như nhau** → chọn cái đơn giản hơn |
| Chênh lệch lớn hơn nhiều lần std | Khác biệt thật |
| Model phức tạp không hơn baseline | Không đáng đưa vào production |

> ⚠️ Lỗi rất phổ biến: chọn model phức tạp vì hơn 0.3% mà **không nhìn std**.

**Về Gradient Boosting:** cây sau tập trung sửa lỗi của cây trước.
```
Cây 1 → sai ở đâu? → Cây 2 sửa chỗ đó → Cây 3 sửa tiếp → …
```
Ở đây dùng `HistGradientBoostingClassifier` của sklearn (rất nhanh, không cần cài thêm). **XGBoost** và **LightGBM** cùng ý tưởng, thường mạnh hơn chút và có nhiều tuỳ chọn hơn — cài sẵn trong nhóm `[ml]`.

## 5. Feature importance

In [ ]:
rf_model = rf.named_steps["m"]
imp = pd.Series(rf_model.feature_importances_, index=ten_dac_trung).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imp.index, imp.to_numpy(), color=MAU_CHINH, height=.65)
ax.set_xlabel("mức độ quan trọng")
ax.set_title("Feature importance — Random Forest")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**Dùng để làm gì:**
- **Phát hiện leakage** — một cột chiếm gần hết importance là cờ đỏ
- Bỏ bớt đặc trưng vô dụng → model nhẹ và ổn định hơn
- Giải thích cho người dùng nghiệp vụ

> ⚠️ **Ba cảnh báo:**
> 1. **Importance ≠ nhân quả.** Model nói `so_lan_ho_tro` quan trọng **không** có nghĩa chặn tổng đài sẽ giữ được khách. (Bài học tương quan/nhân quả từ Phase 02, quay lại ở tầng cao hơn.)
> 2. Importance mặc định của cây **thiên vị** đặc trưng có nhiều giá trị khác nhau.
> 3. Với hai đặc trưng tương quan cao, importance bị **chia đôi** — cả hai trông có vẻ ít quan trọng.

**Permutation importance** khắc phục được các vấn đề này:

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(rf, X_test, y_test, n_repeats=5,
                            random_state=42, scoring="roc_auc", n_jobs=-1)
pi_s = pd.Series(pi.importances_mean, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(pi_s.index, pi_s.to_numpy(), xerr=pi.importances_std[pi_s.index.map(
    lambda c: list(X.columns).index(c))], color=MAU[2], height=.6,
    error_kw={"ecolor": "#444", "capsize": 3})
ax.set_xlabel("AUC giảm bao nhiêu khi xáo trộn cột này")
ax.set_title("Permutation importance (đo trên tập TEST)")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**Cách hoạt động:** xáo trộn ngẫu nhiên một cột rồi đo xem điểm số **giảm bao nhiêu**. Giảm nhiều = cột đó quan trọng.

**Ưu điểm:** đo trên tập **test**, dùng được với **mọi** model (kể cả LLM!), và trả lời đúng câu hỏi *"cột này đóng góp gì vào hiệu năng thật?"*

> 🔗 Ý tưởng "xoá một thành phần rồi đo xem chất lượng giảm bao nhiêu" chính là **ablation study** — bạn sẽ dùng nó ở Phase 08 để biết reranking có thật sự đáng tiền không.

## 6. Tinh chỉnh siêu tham số

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

luoi = {
    "m__max_depth": [3, 5, 8, 12, None],
    "m__min_samples_leaf": [1, 5, 20, 50],
    "m__max_features": ["sqrt", "log2", 0.5],
}

tim = RandomizedSearchCV(
    Pipeline([("pre", pre), ("m", RandomForestClassifier(n_estimators=100,
                                                         random_state=42, n_jobs=-1))]),
    luoi, n_iter=12, cv=3, scoring="roc_auc", random_state=42, n_jobs=-1,
).fit(X_train, y_train)

print("Tham số tốt nhất:")
for k, v in tim.best_params_.items():
    print(f"  {k:<22} {v}")
print(f"\n  AUC (CV)   : {tim.best_score_:.4f}")
print(f"  AUC (test) : {roc_auc_score(y_test, tim.predict_proba(X_test)[:, 1]):.4f}")

**`RandomizedSearchCV` vs `GridSearchCV`:** grid thử **mọi** tổ hợp (bùng nổ nhanh: 5×4×3 = 60 tổ hợp × 3 fold = 180 lần train). Random thử `n_iter` tổ hợp ngẫu nhiên — với cùng ngân sách tính toán, nó thường tìm được kết quả **tốt hơn** vì khám phá được nhiều giá trị hơn cho từng tham số.

> ⚠️ **Thứ tự ưu tiên khi muốn cải thiện model:**
> ```
> ① Thêm dữ liệu             ← hiệu quả nhất
> ② Thêm đặc trưng tốt        ← thường là bước ăn điểm nhất
> ③ Làm sạch dữ liệu kỹ hơn
> ④ Đổi loại model
> ⑤ Tinh chỉnh siêu tham số   ← hiệu quả THẤP NHẤT
> ```
> Người mới thường bắt đầu từ ⑤ và dừng ở đó. Tinh chỉnh thường chỉ mang lại 1–2%, trong khi một đặc trưng đúng có thể mang lại 10%.

## 7. Bảng chọn model

| Model | Dùng khi | Tránh khi |
|---|---|---|
| **Logistic/Linear** | Cần giải thích, dữ liệu ít, quan hệ tuyến tính | Quan hệ phi tuyến phức tạp |
| **Decision Tree** | Cần vẽ ra cho người không kỹ thuật | Cần độ chính xác cao |
| **Random Forest** | Mặc định an toàn, ít phải chỉnh | Cần suy luận cực nhanh |
| **Gradient Boosting** | Cần điểm cao nhất trên dữ liệu **bảng** | Dữ liệu rất ít, cần giải thích |
| **Neural network** | Ảnh, văn bản, âm thanh | **Dữ liệu dạng bảng** |

**Thứ tự nên thử:**
```
① Baseline (Dummy)
② Logistic/Linear Regression      ← đơn giản, giải thích được
③ Random Forest                   ← mạnh, ít phải chỉnh
④ Gradient Boosting               ← mạnh nhất với dữ liệu bảng
```
Chỉ đi tiếp khi bước trước thật sự không đủ.

> 💡 **Trong sản xuất, một Logistic Regression 0.84 giải thích được thường TỐT HƠN một XGBoost 0.86 không ai hiểu.** *"Vì sao khách này bị từ chối vay?"* là câu hỏi phải trả lời được — đôi khi còn là yêu cầu pháp lý.

## ✍️ Tiếp theo

Bạn đã có đủ mọi thứ cho project. Xong bài này → làm [**P2 — Churn Prediction**](../../../projects/P2-churn-prediction/README.md).